# Données

## Importation des packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 500)

import requests
from bs4 import BeautifulSoup
import os
import s3fs
import ast
import json

## Lecture des fichiers movies_metadata.csv et credits.csv

Les données de movies_metadata.csv et credits.csv sont des données trouvées sur Kaggle qui centralisent des informations diverses sur des films sortis avant juillet 2017. 

Les variables de movies_metadata.csv sont :
- adult : signification de la variable non connue
- belongs to collection : si le film appartient à une série de film, la variable renseigne les films faisant partie de cette série
- budget : budget du film
- genres : genres du film
- homepage : lien vers le site officiel du film s'il y en a un
- id et imdb_id : identifiants du film
- original_language : langue originale du film
- original_title : titre original du film
- overview : résumé du film
- popularity : popularité du film sur IMDB
- poster_path : lien vers l'affiche du film
- production_countries : pays de production du film
- production_companies : compagnies de production du film
- release_date : date de sortie du film
- revenue : recettes du film
- runtime : durée du film
- spoken_languages : langues parlées dans le film en version originale
- status : si le film est sorti, prévu, annulé, en production etc
- tagline : catchphrase du film
- title : titre anglophone du film
- video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Les variables de credits.csv sont :
- cast : casting du film sous forme de liste de dictionnaires
- crew : équipe du film sous forme de liste de dictionnaires

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera ien reçu par le public et ainsi remarquer (ou non) la prévisibilité du succès d'un film.

Nous pourrons pondérer l'erreur de prévision avec la variable vote_count et faire de la classification non supervisée dans les stats descriptives

In [2]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_51998/2309787388.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [3]:
FILE_KEY_S3 = '/credits.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_credits = pd.read_csv(file_in,sep=',', header=0)

## Retraitements

Après première exploration des données movies_metadata, nous avons décidé d'enlever les variables suivantes : adult, homepage, overview, popularity, poster_path, spoken_languages et tagline. En effet, la variable adult présente presque toujours la modalité False et semble avoir peu d'intérêt. La variable homepage renseigne le lien vers le site officiel du film s'il y en a un. La variable overview contient les résumés des films du dataframe, ce qui peut être intéressant à exploiter mais nous avons décidé de ne pas le faire. La variable popularity est la popularité du film sur IMDB au moment où les données ont été extraites, c'est donc une variable qui n'est pas statique et qui est calculée directement par IMDB d'une façon que nous ignorons donc nous ne souhaitons pas la prendre en compte. La variable poster_path indique le lien vers l'affiche du film, nous n'en avons pas besoin. La variable spoken_languages indique les langues parlées durant le film en version originale, nous considérons que cette variable est redondante par rapport à la variable original_language. Enfin la variable tagline indique la catchphrase du film, ce qui est à nos yeux peu utile également.

Nous allons retraiter certaines variables. Par exemple, la variable belongs_to_collection sera transformée en booléen (1 si le film correspond à une série de films, 0 sinon) à laquelle nous ajouterons une variable avec le nombre de films précédents de la série ainsi que la note du film précédent. La variable genres sera décomposée en plusieurs variables genre_1, genre_2 etc. Ce genre de décomposition sera également nécessaire pour les variables production_countries et production_companies

Les variables budget et runtime présentent des valeurs manquantes, que nous allons essayer de compléter avec du web scraping.

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False)

Les données du fichier credits.csv vont nous permettre d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. Nous souhaitons ajouter des variables relatives à la popularité des acteurs et du réalisateur via du web scraping.

Nous n'allons pas prendre en compte la variable revenue dans l'étude Machine Learning car nous souhaitons être capables de prédire la note d'un film qui ne serait pas encore sorti, donc les recettes du film ne sont pas connues à l'avance.


In [4]:

data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df[data_movies_df['vote_count'] > 0]
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'spoken_languages', 'tagline', 'status', 'video'])
data_movies_df = data_movies_df.dropna(subset= ['release_date'])
data_movies_df = data_movies_df.dropna(subset= ['imdb_id'])
data_movies_df = data_movies_df.dropna(subset= ['original_language'])
data_movies_df= data_movies_df.drop_duplicates()
data_movies_df = data_movies_df.groupby('id', group_keys=False).apply(lambda group: group.loc[group['vote_count'] == group['vote_count'].min()]).reset_index(drop=True)

/tmp/ipykernel_51998/913713924.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_movies_df = data_movies_df.groupby('id', group_keys=False).apply(lambda group: group.loc[group['vote_count'] == group['vote_count'].min()]).reset_index(drop=True)


In [5]:
data_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   belongs_to_collection  4402 non-null   object 
 1   budget                 42028 non-null  object 
 2   genres                 42028 non-null  object 
 3   id                     42028 non-null  object 
 4   imdb_id                42028 non-null  object 
 5   original_language      42028 non-null  object 
 6   original_title         42028 non-null  object 
 7   production_companies   42028 non-null  object 
 8   production_countries   42028 non-null  object 
 9   release_date           42028 non-null  object 
 10  revenue                42028 non-null  float64
 11  runtime                41879 non-null  float64
 12  title                  42028 non-null  object 
 13  vote_average           42028 non-null  float64
 14  vote_count             42028 non-null  float64
dtypes:

### Retraitement des variables budget et runtime

La variable budget a été reconnue comme chaîne de caractères par Python, nous la convertissons donc en numeric. Pour les variables budget et runtime lorsque la valeur n'est pas connue c'est un 0 qui s'affiche, nous remplaçons donc tous les 0 par NaN.

In [6]:
# conversion de "budget" en nombre
#data_movies_df["budget"] = pd.to_numeric(data_movies_df["budget"], errors="coerce")

# budget et duree (runtime) nuls a considerer comme valeurs manquantes
data_movies_df["budget"] = data_movies_df["budget"].replace('0', np.nan)
data_movies_df["runtime"] = data_movies_df["runtime"].replace(0, np.nan)

In [7]:
df=data_movies_df

### Retraitement de la variable belongs_to_collection

Nous souhaitons ajouter un booléen qui vaut 1 si le film fait partie d'une saga de films et 0 sinon, le nom de la saga à laquelle il appartient, le rang du film dans la saga ainsi que le nombre de films que contient la saga au total et la note du film précédent dans la saga.  

In [8]:
# belongs to collection : 
# on crée un booleen indic_collec 
df["indic_collec"]  = df["belongs_to_collection"].notna().astype("int8") # int8 → 1 octet par valeur
df["indic_collec"].value_counts()
df_collec = df[df["indic_collec"] == 1]
df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)

# on cree une fonction pour recuperer le nom de la collection s il est rempli, rien sinon
def recup_collec(x):
    if pd.notna(x):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return None
    return x

df['belongs_to_collection'] = df['belongs_to_collection'].apply(recup_collec)
df['nom_collec'] = df['belongs_to_collection'].apply(
            lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
                                                )

/tmp/ipykernel_51998/4206164378.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)


In [9]:
# creation d'une colonne rang
# on trie par collec et date de sortie
df = df.sort_values(by=['nom_collec', 'release_date'], ascending=[True, True])
df["rang"] = np.nan # initialisation
masque = df["nom_collec"].notna() # si collec n'est pas manquant
df.loc[masque, "rang"] = df.loc[masque].groupby("nom_collec").cumcount() + 1
#df.rang.value_counts(dropna=False)

In [10]:
# on rajoute une colonne rang max
df_max_rang = df.groupby('nom_collec')['rang'].max().reset_index()
df_max_rang.rename(columns={'rang': 'max_rang'}, inplace=True)
df = df.merge(df_max_rang, on='nom_collec', how='left')

In [11]:
#  ceux qui n'ont qu'un seul opus present sont a considerer comme ne faisant pas partie d'une serie
condition = (df['rang'] == 1) & (df['max_rang'] == 1)

df.loc[condition, 'indic_collec'] = 0
df.loc[condition, 'nom_collec'] = None
df.loc[condition, 'rang'] = np.nan
df.loc[condition, 'max_rang'] = np.nan

#df['max_rang']

In [12]:
# pour les sagas, on recupere la note de l opus precedent
df_prec = df[df['rang'] >= 1]

df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df
df_prec = df_prec.rename(columns={'vote_average': 'vote_prec'}) # colonne vote_average renommée poru etre gardee lors de la fusion
df_prec = df_prec[['rang','nom_collec','vote_prec']]


# Merge sur nom_collec et rang
df2 = df.merge(df_prec[['nom_collec', 'rang', 'vote_prec']],
                     on=['nom_collec', 'rang'],
                     how='left')

df = df2.drop(columns=['belongs_to_collection'])

/tmp/ipykernel_51998/2582065488.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df


In [13]:
df.shape

(42028, 19)

### Retraitement de la variable genres

In [14]:
# variable genres : a considerer comme une liste de dictionnaire
df['genres'] = df['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_genres = df['genres'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
#print(f"Nombre maximum de dictionnaires dans 'genres' : {max_genres}")


In [15]:
# Creation des colonnes genre1 à genre8
for i in range(8):
    col_name = f'genre{i+1}'
    df[col_name] = df['genres'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

### Retraitement de la variable production_companies

In [16]:
# variable production_companies : a considerer comme une liste de dictionnaire
df['production_companies'] = df['production_companies'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_companies'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
#print(f"Nombre maximum de dictionnaires dans 'production_companies' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 26 colonnes

In [17]:
# Creation des colonnes prod1 à prod26
for i in range(26):
    col_name = f'prod{i+1}'
    df[col_name] = df['production_companies'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

### Retraitement de la variable production_countries

In [18]:
# variable production_countries : a considerer comme une liste de dictionnaire
df['production_countries'] = df['production_countries'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_countries'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
print(f"Nombre maximum de dictionnaires dans 'production_countries' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 25 colonnes

Nombre maximum de dictionnaires dans 'production_countries' : 25


In [19]:
# Creation des colonnes pays1 a pays25
for i in range(25):
    col_name = f'pays{i+1}'
    df[col_name] = df['production_countries'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [20]:
missing_percentage = df.isna().sum()

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
pays22        42027
pays25        42027
pays23        42027
pays24        42027
pays13        42027
pays17        42027
pays16        42027
pays15        42027
pays14        42027
pays18        42027
pays20        42027
pays19        42027
pays21        42027
pays12        42026
genre8        42025
prod26        42025
prod25        42024
pays11        42024
pays10        42023
prod23        42023
prod24        42023
prod22        42020
pays9         42018
prod21        42016
pays8         42012
prod20        42011
prod19        42006
prod18        42003
genre7        42002
pays7         42001
prod17        41996
prod16        41976
prod15        41967
pays6         41963
prod14        41952
prod13        41933
prod12        41899
prod11        41857
genre6        41848
pays5         41817
prod10        41790
prod9         41658
prod8         41483
pays4         41367
prod7         41176
genre5        41036
prod6         40623
pays3         39963
prod5         39690
vot

In [21]:
df.shape

(42028, 78)

In [22]:
data_movies_df=df

### Web scraping Wikipédia

Nous souhaitons compléter la variable budget qui présente beaucoup de valeurs manquantes, ainsi qu'ajouter des informations sur les acteurs et réalisateurs.

On ajoute au dataframe les différents url wikipédia possibles pour un film (selon le nom du film, il faut parfois ajouter film ou film + année de sortie à l'url wikipédia pour tomber sur la bonne page wiki)

In [23]:
url_wikipedia_fr = "https://fr.wikipedia.org/wiki/"
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
data_movies_df['url'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_")
data_movies_df['url_film'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film)"
data_movies_df['release_year'] = data_movies_df.release_date.str[:4]
data_movies_df['url_film_date'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film,_" + data_movies_df.release_year + ")"
data_movies_df['id'] = pd.to_numeric(data_movies_df['id'])


On joint les dataframes movies et credits pour ajouter le casting et l'équipe du film.

In [25]:
data_movies_credits = data_movies_df.merge(data_credits.drop_duplicates(), left_on='id', right_on='id', how='left')
data_movies_credits


,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,revenue,...,pays22,pays23,pays24,pays25,url,url_film,release_year,url_film_date,cast,crew
0,70000000,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...",117263,tt2302755,en,Olympus Has Fallen,"[{'name': 'Nu Image Films', 'id': 925}, {'name...","[{'iso_3166_1': 'US', 'name': 'United States o...",2013-03-20,161025640.0,...,None,None,None,None,https://en.wikipedia.org/wiki/Olympus_Has_Fallen,https://en.wikipedia.org/wiki/Olympus_Has_Fall...,2013,https://en.wikipedia.org/wiki/Olympus_Has_Fall...,"[{'cast_id': 7, 'character': 'Mike Banning', '...","[{'credit_id': '52fe4bafc3a36847f820f62f', 'de..."
1,60000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",267860,tt3300542,en,London Has Fallen,"[{'name': 'Millennium Films', 'id': 10254}, {'...","[{'iso_3166_1': 'BG', 'name': 'Bulgaria'}, {'i...",2016-03-02,205754447.0,...,None,None,None,None,https://en.wikipedia.org/wiki/London_Has_Fallen,https://en.wikipedia.org/wiki/London_Has_Falle...,2016,https://en.wikipedia.org/wiki/London_Has_Falle...,"[{'cast_id': 3, 'character': 'Mike Banning', '...","[{'credit_id': '5363c191c3a368157d002773', 'de..."
2,NaN,"[{'id': 35, 'name': 'Comedy'}, {'id': 80, 'nam...",1652,tt0109000,de,00 Schneider - Jagd auf Nihil Baxter,"[{'name': 'Senator Film Produktion', 'id': 191}]","[{'iso_3166_1': 'DE', 'name': 'Germany'}]",1994-12-21,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/00_Schneider_-_J...,https://en.wikipedia.org/wiki/00_Schneider_-_J...,1994,https://en.wikipedia.org/wiki/00_Schneider_-_J...,"[{'cast_id': 2, 'character': '00 Schneider / N...","[{'credit_id': '52fe4309c3a36847f80358a9', 'de..."
3,NaN,"[{'id': 10752, 'name': 'War'}, {'id': 18, 'nam...",19430,tt0046671,de,08/15,[],"[{'iso_3166_1': 'DE', 'name': 'Germany'}]",1954-09-30,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/08/15,https://en.wikipedia.org/wiki/08/15_(film),1954,"https://en.wikipedia.org/wiki/08/15_(film,_1954)","[{'cast_id': 1, 'character': 'Gefreiter Asch',...","[{'credit_id': '52fe47d89251416c750a758d', 'de..."
4,NaN,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",10035,tt0214388,en,100 Girls,"[{'name': 'Dream Entertainment', 'id': 1630}]","[{'iso_3166_1': 'US', 'name': 'United States o...",2000-09-01,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/100_Girls,https://en.wikipedia.org/wiki/100_Girls_(film),2000,"https://en.wikipedia.org/wiki/100_Girls_(film,...","[{'cast_id': 16, 'character': 'Matthew', 'cred...","[{'credit_id': '52fe430e9251416c75001dbd', 'de..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42028,NaN,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",374471,tt4446472,en,Porto,[],"[{'iso_3166_1': 'PL', 'name': 'Poland'}, {'iso...",2017-09-14,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/Porto,https://en.wikipedia.org/wiki/Porto_(film),2017,"https://en.wikipedia.org/wiki/Porto_(film,_2017)","[{'cast_id': 7, 'character': 'Mati Vargnier', ...","[{'credit_id': '568024559251412e5200b981', 'de..."
42029,4696772,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",398818,tt5726616,en,Call Me by Your Name,"[{'name': 'Sony Pictures Classics', 'id': 58},...","[{'iso_3166_1': 'BR', 'name': 'Brazil'}, {'iso...",2017-10-27,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/Call_Me_by_Your_...,https://en.wikipedia.org/wiki/Call_Me_by_Your_...,2017,https://en.wikipedia.org/wiki/Call_Me_by_Your_...,"[{'cast_id': 4, 'character': 'Elio Perlman', '...","[{'credit_id': '5742ea0ec3a3686c8c001a6e', 'de..."
42030,NaN,"[{'id': 99, 'name': 'Documentary'}]",359749,tt4372240,en,78/52,"[{'name': 'Exhibit A Pictures', 'id': 63267}]","[{'iso_3166_1': 'US', 'name': 'United States o...",2017-11-03,0.0,...,None,None,None,None,https://en.wikipedia.org/wiki/78/52,https://en.wikipedia.org/wiki/78/52_(film),2017,"https://en.wikipedia.org/wiki/78/52_(film,_2017)","[{'cast_id': 9, 'char

In [26]:
data_movies_credits.shape

(42033, 84)

On retraite les colonnes cast et crew pour que Python les reconnaissent en tant que liste de dictionnaires.

In [29]:
data_movies_credits = data_movies_credits.dropna(subset= ['cast'])

data_movies_credits['cast'] = data_movies_credits['cast'].apply(ast.literal_eval)
data_movies_credits['crew'] = data_movies_credits['crew'].apply(ast.literal_eval)

On ajoute les colonnes correspondant aux 4 acteurs principaux du film et une colonne pour le réalisateur du film.

In [30]:

data_movies_credits['acteur1'] = data_movies_credits['cast'].apply(
    lambda lst: lst[0]['name'] if isinstance(lst, list) and len(lst) > 0 else None
)
data_movies_credits['acteur2'] = data_movies_credits['cast'].apply(
    lambda lst: lst[1]['name'] if isinstance(lst, list) and len(lst) > 1 else None
)
data_movies_credits['acteur3'] = data_movies_credits['cast'].apply(
    lambda lst: lst[2]['name'] if isinstance(lst, list) and len(lst) > 2 else None
)

data_movies_credits['acteur4'] = data_movies_credits['cast'].apply(
    lambda lst: lst[3]['name'] if isinstance(lst, list) and len(lst) > 3 else None
)
data_movies_credits['realisateur'] = data_movies_credits['crew'].apply(
    lambda lst: lst['job' == 'Director']['name'] if isinstance(lst, list) and len(lst) > 0 else None
)


data_movies_credits = data_movies_credits.drop(columns=['cast', 'crew', 'genres', 'production_companies', 'production_countries'])


In [39]:
data_movies_credits = data_movies_credits.drop_duplicates()
indices_a_exclure = [17411, 23836, 35546, 39616]
data_movies_credits = data_movies_credits.drop(indices_a_exclure)

In [40]:
data_movies_credits.shape

(42028, 84)

In [ ]:
data_movies_credits[data_movies_credits["budget"].isna()].shape

(33265, 84)

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}

Fonction pour trouver le budget d'un film avec l'url wikipédia

In [ ]:
def extraire_budget_depuis_wikipedia(url):
    try:
        # Charger la page
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parser le HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Trouver l'infobox (il peut y avoir plusieurs classes, mais 'infobox' est souvent commun)
        infobox = soup.find('table', class_='infobox')

        if infobox is None:
            return None  # Pas d'infobox trouvée

        # Chercher les lignes de l'infobox
        rows = infobox.find_all('tr')

        for row in rows:
            header = row.find('th')
            if header and 'budget' in header.get_text(strip=True).lower():
                # Trouver la cellule contenant la valeur
                value_cell = row.find('td')
                if value_cell:
                    return value_cell.get_text(separator=" ", strip=True)

        return None  # Pas de ligne contenant "budget"

    except Exception as e:
        print(f"Erreur lors du traitement de {url}: {e}")
        return None


In [ ]:
data['budget_url'] = data['url'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film'] = data['url_film'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film_date'] = data['url_film_date'].apply(extraire_budget_depuis_wikipedia)

# Ton DataFrame à sauvegarder
# Exemple : df = pd.DataFrame({'col1': [1, 2], 'col2': ['a', 'b']})
BUCKET = 'mlepennec-ensae'

FILE_OUT_S3 = '/movies_budget.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    data.to_csv(f_out, index=False)

In [44]:
FILE_KEY_S3 = '/movies_budget.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_budget = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_51998/1137755944.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  data_budget = pd.read_csv(file_in,sep=',', header=0)


In [51]:
df=data_movies_credits.merge(data_budget[["id", "budget_url", "budget_url_film", "budget_url_film_date"]].drop_duplicates(),how='left', left_on='id', right_on='id')


In [52]:
df.shape

(42028, 87)

In [55]:

# Création de la colonne 'budget_final'
df['budget_final'] = (
    df['budget']
    .fillna(df["budget_url"])
    .fillna(df['budget_url_film'])
    .fillna(df['budget_url_film_date'])
)


In [56]:
FILE_OUT_S3 = '/movies_credits_budget.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    df.to_csv(f_out, index=False)

### Web scraping IMDB

In [ ]:


response = requests.get('https://www.imdb.com/fr/title/tt3859310/', headers=headers)
response.status_code
        # Parser le HTML
soup = BeautifulSoup(response.content, 'html.parser')
selector="#__next > main > div > section.ipc-page-background.ipc-page-background--base.sc-358297d7-0.CHcbB > div > section > div > div.sc-e1aae3e0-1.eEFIsG.ipc-page-grid__item.ipc-page-grid__item--span-2 > section:nth-child(47) > div.sc-314065ad-0.hZXevt > ul > li > div > ul > li > span"
print(soup.select(selector)[0].text)

In [ ]:
condition=(df['pays1']=="United States of America") & (df["budget_final"].isna())
df.loc[condition]